# 01 — Préparer les features métier

Cette refonte crée **neuf features candidates**, à partir de mécanismes explicités : ancienneté, désengagement, support, satisfaction connue et variation tarifaire. Une plausibilité métier ne démontre pas un gain prédictif.

Une seule classe publique, `ChurnFeatureEngineer`, orchestre les petites transformations. Aucun modèle prédictif, aucune sélection par score de test et aucun entraînement MLflow ici. Les données sources et les anciens runs restent conservés.

Exécuter les cellules dans l'ordre. La préparation v2 est consommée par le 02, qui l'archive dans MLflow avec le modèle.

## 1 — Imports et paramètres

Les périodes et chemins sont définis avant les calculs.

In [ ]:
import hashlib
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.model_selection import GroupShuffleSplit

if not Path("settings.py").is_file() and Path("../settings.py").is_file():
    sys.path.insert(0, str(Path.cwd().parent))
from settings import settings

SEED = 42
TRAIN_MONTHS = ("2024-01", "2024-07")
VALIDATION_MONTHS = ("2024-08", "2024-09")
TEST_MONTHS = ("2024-10", "2024-11")
WALK_FORWARD_MONTHS = ("2024-08", "2024-11")
PREPARATION_DIR = settings.FEATURES_DIR
NOTEBOOK_PATH = settings.PROJECT_ROOT / "notebooks" / "01_churn_feature_engineering.ipynb"

## 2 — Lire, normaliser et auditer

Montants et compteurs négatifs, valeurs non finies et NPS hors 0–10 deviennent manquants dans la copie de travail. Les copies normalisées sont dédupliquées ; toutes les clés ambiguës sont exclues. Aucune imputation statistique.

In [ ]:
def read_telco_source(path):
    """Normaliser une copie ; tracer les exclusions sans modifier le CSV source."""
    raw = pd.read_csv(path, dtype={"customer_id": "string", "dept": "string"})
    required = {
        "customer_id",
        "snapshot_date",
        "signup_date",
        "contract",
        "region",
        "dept",
        "channel",
        "monthly_fee",
        "data_gb",
        "support_calls_3m",
        "late_payments",
        "nps",
        "churn",
    }
    missing = required - set(raw.columns)
    if missing:
        raise ValueError(f"Colonnes absentes : {sorted(missing)}")
    data = raw.copy()
    for column in ["customer_id", "contract", "region", "dept", "channel"]:
        data[column] = data[column].astype("string").str.strip().str.strip("\"'")
    for column in ["contract", "region", "dept", "channel"]:
        data[column] = data[column].str.lower().replace("", pd.NA)
    for column in ["snapshot_date", "signup_date"]:
        data[column] = pd.to_datetime(data[column], errors="coerce")
    if (
        data.customer_id.isna().any()
        or data.customer_id.eq("").any()
        or data.snapshot_date.isna().any()
    ):
        raise ValueError("Identifiant ou snapshot_date manquant/invalide.")
    numeric = ["monthly_fee", "data_gb", "support_calls_3m", "late_payments", "nps", "churn"]
    numeric += [c for c in ["age", "last_invoice_amount", "retention_offer_declined"] if c in data]
    invalid_rows = []
    for column in numeric:
        values = pd.to_numeric(data[column], errors="coerce")
        invalid = data[column].notna() & values.isna()
        invalid |= values.notna() & ~np.isfinite(values)
        if column in [
            "monthly_fee",
            "data_gb",
            "support_calls_3m",
            "late_payments",
            "last_invoice_amount",
        ]:
            invalid |= values.lt(0)
        if column == "nps":
            invalid |= values.notna() & ~values.between(0, 10)
        if column == "churn":
            if invalid.any() or not values.dropna().isin([0, 1]).all():
                raise ValueError("La cible doit valoir 0, 1 ou être manquante.")
        data[column] = values.mask(invalid)
        invalid_rows.append({"column": column, "invalid_values": int(invalid.sum())})
    before = len(data)
    data = data.drop_duplicates().copy()
    copies_removed = before - len(data)
    ambiguous = data.duplicated(["customer_id", "snapshot_date"], keep=False)
    ambiguous_keys = data.loc[ambiguous, ["customer_id", "snapshot_date"]].drop_duplicates()
    data = data.loc[~ambiguous].copy()
    if data.empty:
        raise ValueError("Aucune observation non ambiguë.")
    if (
        data.assign(month=data.snapshot_date.dt.to_period("M"))
        .duplicated(["customer_id", "month"])
        .any()
    ):
        raise ValueError("Plusieurs snapshots dans le même mois pour un client.")
    quality = {
        "raw_rows": len(raw),
        "retained_rows": len(data),
        "normalized_copies_removed": copies_removed,
        "ambiguous_keys_excluded": len(ambiguous_keys),
        "missing_labels": int(data.churn.isna().sum()),
    }
    return raw, data, pd.DataFrame(invalid_rows), quality


raw_df, data, invalid_values, quality = read_telco_source(settings.DATA_PATH)
display(pd.Series(quality, name="value").to_frame(), invalid_values)

## 3 — Contrat métier des nouvelles variables

| Feature | Calcul | Interprétation et limite |
|---|---|---|
| `tenure_months` | jours(snapshot − inscription) / 30,4375 | Ancienneté ; date absente/future → manquant |
| `usage_change_vs_recent_pct` | 100 × (usage actuel − médiane des 3 mois précédents) / médiane | Écart aux habitudes ; 4 mois consécutifs, mesures valides et référence > 0 requis |
| `zero_usage_streak` | longueur de la séquence mensuelle d'usage nul jusqu'au snapshot | Inactivité observée ; trou de suivi ou valeur manquante interrompt la séquence |
| `support_pressure_change` | appels sur 3 mois au snapshot courant − au précédent | Variation nette d'une fenêtre glissante ; **pas** le nombre de nouveaux appels |
| `support_presence_3_snapshots` | somme des indicateurs appels > 0 sur les 3 derniers snapshots consécutifs | Persistance du support ; fenêtres chevauchantes, pas neuf mois indépendants |
| `last_observed_nps` | dernière note valide du même client, connue au snapshot | Pas de remplissage avec une observation future |
| `months_since_observed_nps` | mois courant − dernier mois avec note valide | Fraîcheur dans le fichier, pas âge réel d'un questionnaire |
| `monthly_fee_change_pct` | 100 × (prix actuel − prix précédent) / prix précédent | Deux mois consécutifs et prix précédent > 0 ; interprétation commerciale à confirmer |
| `monthly_fee_band` | tertiles des prix valides du train | Segment descriptif ; bornes figées ensuite, valeurs absentes séparées |

Les prix varient presque chaque mois dans ce dataset : une variation n'est donc pas automatiquement un changement commercial. Les unités et la période de `data_gb` doivent aussi être confirmées.

La facture, ses dérivées, l'offre de rétention et l'âge restent hors des prédicteurs. Les produits arbitraires, croisements systématiques et seuils non justifiés de la v1 ne sont pas conservés dans cette référence.

## 4 — Définir la classe principale

Les petites classes restent internes. Les calculs temporels sont effectués par client et chronologiquement. Les fenêtres incomplètes restent manquantes.

In [ ]:
class LifecycleFeatures:
    def transform(self, frame):
        result = frame.copy()
        days = (result.snapshot_date - result.signup_date).dt.days
        result["tenure_months"] = (days / 30.4375).where(days.ge(0))
        return result


def monthly_context(frame):
    """Groupes et mois ordinaux du panel déjà trié par la classe principale."""
    clients = frame.customer_id
    month = frame.snapshot_date.dt.year * 12 + frame.snapshot_date.dt.month
    return clients, month


class UsageHistoryFeatures:
    def transform(self, frame):
        result = frame.copy()
        clients, month = monthly_context(result)
        usage = result.data_gb
        previous = [usage.groupby(clients).shift(k) for k in [1, 2, 3]]
        complete = usage.notna()
        for k, values in enumerate(previous, start=1):
            complete &= month.sub(month.groupby(clients).shift(k)).eq(k) & values.notna()
        reference = pd.concat(previous, axis=1).median(axis=1, skipna=False)
        result["usage_change_vs_recent_pct"] = (
            (100 * (usage - reference) / reference.where(reference.gt(0)))
            .where(complete)
            .replace([np.inf, -np.inf], np.nan)
        )
        consecutive = month.sub(month.groupby(clients).shift()).eq(1)
        zero = usage.eq(0)
        previous_zero = zero.groupby(clients).shift(fill_value=False)
        starts = zero & (~previous_zero | ~consecutive)
        blocks = starts.cumsum()
        streak = zero.astype(int).groupby(blocks).cumsum()
        result["zero_usage_streak"] = streak.where(zero, 0).where(usage.notna()).astype(float)
        return result


class SupportHistoryFeatures:
    def transform(self, frame):
        result = frame.copy()
        clients, month = monthly_context(result)
        calls = result.support_calls_3m
        previous = calls.groupby(clients).shift()
        consecutive = month.sub(month.groupby(clients).shift()).eq(1)
        result["support_pressure_change"] = calls.sub(previous).where(
            consecutive & calls.notna() & previous.notna()
        )
        complete = calls.notna()
        presence = calls.gt(0).astype(float)
        for k in [1, 2]:
            lagged = calls.groupby(clients).shift(k)
            complete &= month.sub(month.groupby(clients).shift(k)).eq(k) & lagged.notna()
            presence += lagged.gt(0).astype(float)
        result["support_presence_3_snapshots"] = presence.where(complete)
        return result


class SatisfactionHistoryFeatures:
    def transform(self, frame):
        result = frame.copy()
        clients, month = monthly_context(result)
        # Date de présence dans le fichier, pas date du questionnaire réel.
        result["last_observed_nps"] = result.nps.groupby(clients).ffill()
        last_month = month.where(result.nps.notna()).groupby(clients).ffill()
        result["months_since_observed_nps"] = month - last_month
        return result


class PricingFeatures:
    def fit(self, train):
        values = train.monthly_fee.dropna()
        if values.empty:
            raise ValueError("Aucun tarif valide dans le train.")
        cuts = np.unique(values.quantile([1 / 3, 2 / 3]).to_numpy())
        cuts = cuts[(cuts > values.min()) & (cuts < values.max())]
        self.edges_ = np.r_[-np.inf, cuts, np.inf]
        self.labels_ = {
            1: ["Intermédiaire"],
            2: ["Peu coûteux", "Coûteux"],
            3: ["Peu coûteux", "Intermédiaire", "Coûteux"],
        }[len(cuts) + 1]
        return self

    def transform(self, frame):
        result = frame.copy()
        clients, month = monthly_context(result)
        fee = result.monthly_fee
        previous = fee.groupby(clients).shift()
        consecutive = month.sub(month.groupby(clients).shift()).eq(1)
        result["monthly_fee_change_pct"] = (
            (100 * (fee - previous) / previous.where(previous.gt(0)))
            .where(consecutive)
            .replace([np.inf, -np.inf], np.nan)
        )
        result["monthly_fee_band"] = (
            pd.Series(pd.cut(fee, self.edges_, labels=self.labels_), index=result.index)
            .astype("string")
            .fillna("Non renseigné")
        )
        return result


class ChurnFeatureEngineer:
    """Seul point d'entrée public : apprentissage sur train, transformations causales."""

    required = [
        "customer_id",
        "snapshot_date",
        "signup_date",
        "monthly_fee",
        "data_gb",
        "support_calls_3m",
        "nps",
    ]
    created_features = [
        "tenure_months",
        "usage_change_vs_recent_pct",
        "zero_usage_streak",
        "support_pressure_change",
        "support_presence_3_snapshots",
        "last_observed_nps",
        "months_since_observed_nps",
        "monthly_fee_change_pct",
        "monthly_fee_band",
    ]

    def __init__(self):
        self._pricing = PricingFeatures()
        self._steps = [
            LifecycleFeatures(),
            UsageHistoryFeatures(),
            SupportHistoryFeatures(),
            SatisfactionHistoryFeatures(),
            self._pricing,
        ]
        self._history = None

    def fit(self, train):
        self._history = None
        self._pricing.fit(train)
        # Les labels ne font jamais partie de l'historique.
        self._history = train[self.required].copy()
        return self

    def transform(self, frame):
        if self._history is None:
            raise ValueError("Appeler fit sur le train avant transform.")
        keys = ["customer_id", "snapshot_date"]
        if frame[keys].isna().any().any() or frame.duplicated(keys).any():
            raise ValueError("Clé client × snapshot invalide.")
        panel = pd.concat([self._history, frame[self.required]], ignore_index=True)
        panel = panel.drop_duplicates(keys, keep="last").sort_values(keys).reset_index(drop=True)
        if (
            panel.assign(month=panel.snapshot_date.dt.to_period("M"))
            .duplicated(["customer_id", "month"])
            .any()
        ):
            raise ValueError("Plusieurs observations du même client dans un mois.")
        for step in self._steps:
            panel = step.transform(panel)
        selected = panel.set_index(keys).reindex(pd.MultiIndex.from_frame(frame[keys]))
        result = frame.copy()
        for feature in self.created_features:
            result[feature] = selected[feature].to_numpy()
        return result

    def fit_transform(self, train):
        return self.fit(train).transform(train)

    def get_feature_metadata(self):
        if self._history is None:
            raise ValueError("Appeler fit avant de lire les métadonnées.")
        return {
            "feature_set_version": 2,
            "monthly_fee_band": {
                "edges": [float(v) if np.isfinite(v) else None for v in self._pricing.edges_],
                "labels": list(self._pricing.labels_),
                "fit_source": "train_only",
            },
            "temporal_rule": "past_covariates_only; consecutive_months_for_deltas_and_windows",
            "nps_rule": "last_recorded_snapshot_not_survey_date",
        }

## 5 — Ajuster sur train, transformer le contexte disponible

Le contexte inclut les covariables historiques des mois purgés, jamais leurs labels. Une observation future ne modifie aucun calcul passé.

In [ ]:
labeled = data.loc[data.churn.notna()].copy()
train_mask = labeled.snapshot_date.dt.to_period("M").between(*TRAIN_MONTHS)
feature_engineer = ChurnFeatureEngineer()
feature_engineer.fit(labeled.loc[train_mask])
# Garder aussi les snapshots sans label comme historique de covariables.
featured = feature_engineer.transform(data)
featured_labeled = featured.loc[featured.churn.notna()].copy()
NEW_FEATURES = list(feature_engineer.created_features)
ORIGINAL_FEATURES = [
    "contract",
    "region",
    "dept",
    "channel",
    "monthly_fee",
    "data_gb",
    "support_calls_3m",
    "late_payments",
    "nps",
]
MODEL_FEATURES = ORIGINAL_FEATURES + NEW_FEATURES
CATEGORICAL_FEATURES = ["contract", "region", "dept", "channel", "monthly_fee_band"]
EXCLUDED = [c for c in data.columns if c not in ORIGINAL_FEATURES]
assert not set(MODEL_FEATURES) & set(EXCLUDED)
display(featured[NEW_FEATURES].head())

## 6 — Partitions et contrôle des colonnes

Le test est préparé, mais aucun taux ou score de test ne guide les choix. Les anomalies du panel restent visibles.

In [ ]:
period = featured_labeled.snapshot_date.dt.to_period("M")
partitions = {
    name: featured_labeled.loc[period.between(*months)].copy()
    for name, months in [
        ("train", TRAIN_MONTHS),
        ("validation", VALIDATION_MONTHS),
        ("test", TEST_MONTHS),
    ]
}
for name, frame in partitions.items():
    if frame.empty or frame.churn.nunique() != 2:
        raise ValueError(f"Partition {name} vide ou mono-classe.")
assert partitions["train"].snapshot_date.max() < partitions["validation"].snapshot_date.min()
assert partitions["validation"].snapshot_date.max() < partitions["test"].snapshot_date.min()
profile = pd.DataFrame(
    {
        "feature": MODEL_FEATURES,
        "missing_train_pct": [partitions["train"][c].isna().mean() * 100 for c in MODEL_FEATURES],
        "distinct_train": [partitions["train"][c].nunique() for c in MODEL_FEATURES],
    }
)
summary = pd.DataFrame(
    [
        {
            "split": name,
            "rows": len(frame),
            "clients": frame.customer_id.nunique(),
            "first_snapshot": str(frame.snapshot_date.min()),
            "last_snapshot": str(frame.snapshot_date.max()),
        }
        for name, frame in partitions.items()
    ]
)
display(profile, summary)

## 7 — Sauvegarder la préparation et les entrées de diagnostic

Le 02 consomme ces fichiers avec SHA-256 ; il ne recalcule aucune feature. Chaque protocole apprend ses bornes tarifaires sur son propre train.

In [ ]:
def save_frame(frame, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(path, index=False)
    return {
        "path": path.relative_to(settings.PROJECT_ROOT).as_posix(),
        "rows": len(frame),
        "sha256": hashlib.sha256(path.read_bytes()).hexdigest(),
    }


def prepare_diagnostic(train_source, control_source, name):
    engineer = ChurnFeatureEngineer().fit(train_source)
    context = engineer.transform(data)
    folder = PREPARATION_DIR / "diagnostics" / name
    return {
        "datasets": {
            "train": save_frame(context.loc[train_source.index], folder / "train.csv"),
            "validation": save_frame(context.loc[control_source.index], folder / "validation.csv"),
        },
        "feature_metadata": engineer.get_feature_metadata(),
    }


PREPARATION_DIR.mkdir(parents=True, exist_ok=True)
datasets = {
    name: save_frame(frame, settings.PROCESSED_DIR / f"{name}.csv")
    for name, frame in partitions.items()
}
reports = {
    "feature_quality": save_frame(profile, PREPARATION_DIR / "reports" / "feature_quality.csv"),
    "split_summary": save_frame(summary, PREPARATION_DIR / "reports" / "split_summary.csv"),
}
pretest = labeled.loc[labeled.snapshot_date.dt.to_period("M") < pd.Period(TEST_MONTHS[0])]
group_split = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
i, j = next(group_split.split(pretest, groups=pretest.customer_id))
diagnostics = {
    "clients": prepare_diagnostic(pretest.iloc[i], pretest.iloc[j], "clients"),
    "walk_forward": [],
}
months = labeled.snapshot_date.dt.to_period("M")
for month in pd.period_range(*WALK_FORWARD_MONTHS, freq="M"):
    train = labeled.loc[months <= month - 2]
    control = labeled.loc[months == month]
    if train.empty or control.empty or train.churn.nunique() != 2 or control.churn.nunique() != 2:
        continue
    entry = prepare_diagnostic(train, control, f"walk_{month}")
    entry.update(month=str(month), last_train_month=str(month - 2))
    diagnostics["walk_forward"].append(entry)

snapshot = json.loads(NOTEBOOK_PATH.read_text())
for cell in snapshot["cells"]:
    if cell["cell_type"] == "code":
        cell.update(outputs=[], execution_count=None)
source_path = PREPARATION_DIR / "source" / "notebook_01.ipynb"
source_path.parent.mkdir(parents=True, exist_ok=True)
source_path.write_text(json.dumps(snapshot, ensure_ascii=False, indent=1))
contract = {
    "preparation_schema_version": 2,
    "seed": SEED,
    "model_features": MODEL_FEATURES,
    "original_features": ORIGINAL_FEATURES,
    "new_features": NEW_FEATURES,
    "categorical_features": CATEGORICAL_FEATURES,
    "excluded": EXCLUDED,
    "feature_metadata": feature_engineer.get_feature_metadata(),
    "datasets": datasets,
    "diagnostics": diagnostics,
    "reports": reports,
    "quality": quality,
    "source": {
        "path": source_path.relative_to(settings.PROJECT_ROOT).as_posix(),
        "sha256": hashlib.sha256(source_path.read_bytes()).hexdigest(),
    },
    "source_csv_sha256": hashlib.sha256(settings.DATA_PATH.read_bytes()).hexdigest(),
}
contract["preparation_id"] = hashlib.sha256(
    json.dumps(contract, sort_keys=True, ensure_ascii=False).encode()
).hexdigest()
contract_path = PREPARATION_DIR / "feature_contract.json"
contract_path.write_text(json.dumps(contract, ensure_ascii=False, indent=2))
print("Préparation v2 sauvegardée :", contract["preparation_id"])
print("Suite : 02_churn_modelisation.ipynb")